# Data Preparation: Segmentación y Exportación de Jugadas Elegibles

Este notebook implementa el pipeline de preparación de datos para la segmentación y clasificación de secuencias sostenidas de juego a partir de eventos de StatsBomb, de acuerdo con la especificación de `docs/specs/clustering-de-jugadas.md` y el glosario de `CONTEXT.md`.

### Principios clave:
1. **Sin credenciales ni estado externo**: Todo el procesamiento se realiza desde los archivos locales en `datos_statsbomb/`.
2. **Validación de integridad**: Se verifica la coherencia entre claves de eventos, `id` y `match_id`.
3. **Filtro de elegibilidad**: Se seleccionan secuencias con al menos 3 pases completados del equipo protagonista (`possession_team`), descartando pases incompletos y del rival.
4. **Conservación de fidelidad tabular**: Las jugadas se exportan conservando todas las columnas originales, su orden, cronología y datos anidados en formato tabular JSON estándar (`orient="split"`), acompañadas de su `manifiesto.json`.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from src.data_preparation.events import cargar_eventos_partido
from src.data_preparation.export import exportar_jugadas_partido, leer_jugada

# ---------------------------------------------------------
# Configuración del pipeline
# ---------------------------------------------------------
RUTA_ENTRADA = Path("datos_statsbomb")
RUTA_SALIDA = Path("datos_preparados")
MIN_PASES_COMPLETADOS = 3
SOBRESCRIBIR = True  # True para permitir reemplazo en reejecución


## 1. Descubrimiento de archivos de eventos locales

In [2]:
archivos_eventos = sorted(list(RUTA_ENTRADA.glob("competicion_*/temporada_*/eventos/*.json")))
print(f"Total de archivos de eventos locales descubiertos: {len(archivos_eventos)}")
if archivos_eventos:
    print(f"Ejemplo de archivo: {archivos_eventos[0]}")


Total de archivos de eventos locales descubiertos: 1808
Ejemplo de archivo: datos_statsbomb/competicion_73/temporada_108/eventos/3799351.json


## 2. Recorrido ejecutable sobre un partido local

In [3]:
# Seleccionar el primer partido disponible para el recorrido ejecutable
archivo_partido = archivos_eventos[0]
print(f"Cargando eventos desde: {archivo_partido}")

# Extraer identificadores de la ruta
partes = archivo_partido.parts
competicion_id = int([p for p in partes if p.startswith("competicion_")][0].split("_")[1])
temporada_id = int([p for p in partes if p.startswith("temporada_")][0].split("_")[1])
partido_id = int(archivo_partido.stem)

metadatos_partido = {
    "competicion_id": competicion_id,
    "temporada_id": temporada_id,
    "partido_id": partido_id,
    "temporada_nombre": f"Temporada {temporada_id}",
}

# Cargar y validar eventos con integridad referencial
df_eventos = cargar_eventos_partido(archivo_partido)
print(f"Partido {partido_id} cargado con éxito:")
print(f"  • Total de eventos: {len(df_eventos)}")
print(f"  • Columnas: {len(df_eventos.columns)}")
print(f"  • Periodos: {sorted(df_eventos['period'].unique().tolist())}")


Cargando eventos desde: datos_statsbomb/competicion_73/temporada_108/eventos/3799351.json
Partido 3799351 cargado con éxito:
  • Total de eventos: 3068
  • Columnas: 54
  • Periodos: [1, 2]


## 3. Segmentación y exportación de jugadas elegibles

In [4]:
# Exportar jugadas elegibles y generar manifiesto de auditoría
resumen = exportar_jugadas_partido(
    df_eventos=df_eventos,
    ruta_salida=RUTA_SALIDA,
    metadatos_partido=metadatos_partido,
    sobrescribir=SOBRESCRIBIR,
    min_pases=MIN_PASES_COMPLETADOS,
)

print("=" * 65)
print("             RESUMEN DE AUDITORÍA DE EXPORTACIÓN")
print("=" * 65)
print(f"Partido ID:                      {resumen['partido_id']}")
print(f"Total eventos de entrada:        {resumen['total_eventos_entrada']}")
print(f"Jugadas elegibles guardadas:     {resumen['total_jugadas_elegibles']}")
print(f"Total eventos en jugadas:        {resumen['total_eventos_exportados']}")
print(f"Secuencias candidatas descart.:  {resumen['total_candidatas_descartadas']}")
print(f"Eventos no exportados auditados: {len(resumen['eventos_no_exportados'])}")
conciliacion = (resumen['total_eventos_exportados'] + len(resumen['eventos_no_exportados'])) == resumen['total_eventos_entrada']
print(f"Conciliación de eventos exacta:  {conciliacion}")
print("=" * 65)


             RESUMEN DE AUDITORÍA DE EXPORTACIÓN
Partido ID:                      3799351
Total eventos de entrada:        3068
Jugadas elegibles guardadas:     76
Total eventos en jugadas:        2010
Secuencias candidatas descart.:  156
Eventos no exportados auditados: 1058
Conciliación de eventos exacta:  True


## 4. Verificación de integridad y recarga de jugada

In [5]:
# Localizar y releer la primera jugada exportada
dir_partido = RUTA_SALIDA / f"competicion_{competicion_id}" / f"temporada_{temporada_id}" / f"partido_{partido_id}"
ruta_primera_jugada = dir_partido / "jugada_0001.json"

df_jugada = leer_jugada(ruta_primera_jugada)

print(f"Jugada releída: {ruta_primera_jugada.name}")
print(f"  • Eventos en la jugada: {len(df_jugada)}")
print(f"  • Coincidencia exacta de columnas: {list(df_jugada.columns) == list(df_eventos.columns)}")
print(f"  • Equipo protagonista: {resumen['jugadas'][0]['equipo_protagonista']}")
print(f"  • Pases completados del protagonista: {resumen['jugadas'][0]['pases_completados']}")

# Mostrar las primeras acciones de la jugada
cols_vista = [c for c in ["id", "period", "index", "timestamp", "type", "team", "possession"] if c in df_jugada.columns]
df_jugada[cols_vista].head(10)


Jugada releída: jugada_0001.json
  • Eventos en la jugada: 71
  • Coincidencia exacta de columnas: True
  • Equipo protagonista: Guadalajara
  • Pases completados del protagonista: 20


,id,period,index,timestamp,type,team,possession
0,8a42ebe5-fbf9-5e27-83a1-cc4194b82860,1,5,00:00:01.057,"{'id': 30, 'name': 'Pass'}","{'id': 1297, 'name': 'Guadalajara'}",2
1,af753232-b48f-5544-b5f6-8bea22aa5b71,1,6,00:00:02.349,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1297, 'name': 'Guadalajara'}",2
2,49065e57-acc1-5602-8010-12d759694f2c,1,7,00:00:02.349,"{'id': 43, 'name': 'Carry'}","{'id': 1297, 'name': 'Guadalajara'}",2
3,32105e22-c5e6-59ff-8ee9-d54dfb517c2d,1,8,00:00:03.965,"{'id': 30, 'name': 'Pass'}","{'id': 1297, 'name': 'Guadalajara'}",2
4,23ee2fd6-ec68-5775-bba2-6c1c8b137514,1,9,00:00:05.158,"{'id': 17, 'name': 'Pressure'}","{'id': 1287, 'name': 'Atlético San Luis'}",2
5,6c680ada-0983-5af6-855a-bb903b353b63,1,10,00:00:05.361,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1297, 'name': 'Guadalajara'}",2
6,9482cb0a-1f6f-5481-b61f-2d2cc8d764f0,1,11,00:00:05.361,"{'id': 43, 'name': 'Carry'}","{'id': 1297, 'name': 'Guadalajara'}",2
7,ca4a8684-ccee-529b-9edc-2cce2dd89938,1,12,00:00:05.667,"{'id': 30, 'name': 'Pass'}","{'id': 1297, 'name': 'Guadalajara'}",2
8,738a3005-e13c-52db-ad21-ce451680a50c,1,13,00:00:08.226,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 1297, 'name': 'Guadalajara'}",2
9,24f4fae7-6d94-53a7-9a63-2677c7309d83,1,14,00:00:08.226,"{'id': 43, 'name': 'Carry'}","{'id': 1297, 'name': 'Guadalajara'}",2


## 5. Inspección del manifiesto

In [6]:
with open(dir_partido / "manifiesto.json", "r", encoding="utf-8") as f:
    manifiesto = json.load(f)

print(f"Manifiesto para partido {manifiesto['partido_id']}:")
print(f"- Total jugadas elegibles: {manifiesto['total_jugadas_elegibles']}")
print(f"- Primera jugada registrada: {manifiesto['jugadas'][0]['play_id']}")
print(f"- Muestra de motivos de eventos no exportados:")
motivos = pd.Series([e['motivo'] for e in manifiesto['eventos_no_exportados']]).value_counts()
print(motivos)


Manifiesto para partido 3799351:
- Total jugadas elegibles: 76
- Primera jugada registrada: c73_t108_m3799351_p0001
- Muestra de motivos de eventos no exportados:
secuencia_descartada_menos_3_pases    1045
evento_administrativo                   13
Name: count, dtype: int64
